In [1]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values, Json
import logging
import json
import html
import re
import os


In [2]:
import pandas as pd
from pathlib import Path
import re

In [3]:
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
TABLE_NAME   = 'proquestarticles_nite'

# Define the root mapping
OLD_ROOT = r'E:\Callproject\assembled'
NEW_ROOT = r'E:\Callproject\4_raw_files\set1'


conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

In [4]:
# Magic byte signatures
MAGIC_BYTES = {
    'jpg': (b'\xff\xd8\xff',),
    'tif': (b'\x49\x49\x2a\x00', b'\x4d\x4d\x00\x2a'),  # little and big endian
    'png': (b'\x89\x50\x4e\x47',),
    'indd': (b'\x06\x06\xed\xf5\xd8\x1d\x46\xe5',),
}

def detect_filetype(filepath):
    """Return the detected filetype string, or None if unrecognised."""
    try:
        with open(filepath, 'rb') as f:
            header = f.read(8)
        for filetype, signatures in MAGIC_BYTES.items():
            for sig in signatures:
                if header.startswith(sig):
                    return filetype
    except (FileNotFoundError, PermissionError, OSError):
        return None
    return None

In [5]:
# Step 1: Fetch all rows where filetype is NULL
cur.execute("""
    SELECT id, folder, filename
    FROM articles.filelist
    WHERE filetype IS NULL;
""")
rows = cur.fetchall()
print(f"Rows to check: {len(rows)}")

Rows to check: 140309


In [ ]:
# Step 2: Test each file locally and collect updates
updates = []
unrecognised = 0
not_found = 0

for row_id, folder, filename in rows:
    # Replace the old root with the new one
    remapped_folder = folder.replace(OLD_ROOT, NEW_ROOT)
    filepath = os.path.join(remapped_folder, filename)

    detected = detect_filetype(filepath)
    if detected:
        updates.append((detected, row_id))
    elif not os.path.exists(filepath):
        not_found += 1
    else:
        unrecognised += 1

print(f"Files to update:     {len(updates)}")
print(f"Files not found:     {not_found}")
print(f"Unrecognised format: {unrecognised}")

from collections import Counter
print(Counter(t for t, _ in updates))

In [ ]:
# Step 3: Batch UPDATE — run only after reviewing the preview above
from psycopg2.extras import execute_values

execute_values(
    cur,
    """
    UPDATE articles.filelist AS f
    SET filetype = v.filetype
    FROM (VALUES %s) AS v(filetype, id)
    WHERE f.id = v.id;
    """,
    updates,
    template="(%s, %s)"
)
conn.commit()
print(f"Rows updated: {cur.rowcount}")